# 📱 Gerador de QR Codes - Fenômenos de Transporte

Este notebook gera QR Codes para cada capítulo do livro, permitindo:

- 🔗 Acesso rápido ao capítulo no GitHub
- 💬 Chat contextual com IA Qwen
- 📊 Download dos dados do estudo de caso
- 🎥 Links para vídeos complementares

## 🚀 Uso

1. Execute todas as células
2. Os QR Codes serão salvos em `qr/generated/`
3. Para inserir no LaTeX: `\includegraphics[width=2cm]{qr/generated/capX.png}`

## 🔧 Configuração

In [ ]:
# Instalação de dependências (se necessário)
# !pip install qrcode[pil] dashscope python-dotenv

import sys
from pathlib import Path

# Adiciona src ao path para imports
sys.path.insert(0, str(Path.cwd() / 'src'))
sys.path.insert(0, str(Path.cwd() / 'scripts'))

In [ ]:
# Importa o gerador
from generate_qrcodes import CHAPTERS, generate_qrcode, generate_all
import qrcode
from IPython.display import Image, display, Markdown

In [ ]:
# Configurações
OUTPUT_DIR = Path("qr/generated")
QR_SIZE = 10  # 1-40, maior = mais legível mas QR Code maior
INCLUDE_CHAT = True  # Incluir parâmetros para chat com IA

print(f"📁 Saída: {OUTPUT_DIR}")
print(f"📏 Tamanho QR: {QR_SIZE}")
print(f"💬 Chat IA: {'Sim' if INCLUDE_CHAT else 'Não'}")

In [ ]:
# Lista capítulos disponíveis
display(Markdown("### 📚 Capítulos disponíveis\n"))
for cid, info in CHAPTERS.items():
    vol = "I" if info["volume"] == 1 else "II"
    print(f"- `{cid}` | Vol {vol} | {info['title']}")

In [ ]:
# 🎯 Gerar QR Code para um capítulo específico (exemplo: cap4)
chapter_id = "cap4"  # Altere conforme necessário

if chapter_id in CHAPTERS:
    print(f"\n🔄 Gerando QR Code para: {CHAPTERS[chapter_id]['title']}\n")
    
    path = generate_qrcode(
        chapter_id=chapter_id,
        output_dir=OUTPUT_DIR,
        size=QR_SIZE,
        include_chat=INCLUDE_CHAT
    )
    
    # Exibe o QR Code gerado
    display(Markdown(f"**QR Code para {chapter_id}:**"))
    display(Image(filename=str(path), width=200))
    
    # Mostra como usar no LaTeX
    display(Markdown(f"""
```latex
% Inserir no capítulo {chapter_id}
\\begin{{center}}
  \\includegraphics[width=2cm]{{{path}}}
  \\[1ex]
  \\small Scan para recursos interativos
\\end{{center}}
```
"""))
else:
    print(f"❌ Capítulo '{chapter_id}' não encontrado. Use a lista acima.")

In [ ]:
# 🚀 Gerar QR Codes para TODOS os capítulos
print("\n🔄 Gerando QR Codes para todos os capítulos...\n")

paths = generate_all(
    output_dir=OUTPUT_DIR,
    size=QR_SIZE,
    include_chat=INCLUDE_CHAT
)

print(f"\n✅ Sucesso! {len(paths)} QR Codes gerados em `{OUTPUT_DIR}/`")
print("\n📋 Lista de arquivos:")
for p in sorted(paths):
    print(f"  - {p.name}")

In [ ]:
# 🔍 Testar leitura de um QR Code (opcional)
# Requer: pip install pyzbar opencv-python

try:
    import cv2
    from pyzbar import pyzbar
    import json
    
    def read_qrcode(image_path: Path):
        img = cv2.imread(str(image_path))
        decoded = pyzbar.decode(img)
        if decoded:
            payload = json.loads(decoded[0].data.decode('utf-8'))
            return payload
        return None
    
    # Testa com o primeiro QR gerado
    if paths:
        test_path = paths[0]
        print(f"\n🔍 Lendo QR Code de teste: {test_path.name}")
        
        payload = read_qrcode(test_path)
        if payload:
            display(Markdown("**Conteúdo decodificado:**"))
            print(json.dumps(payload, indent=2, ensure_ascii=False))
        else:
            print("⚠️ Não foi possível decodificar. Instale: pip install pyzbar opencv-python")
            
except ImportError:
    print("\n💡 Dica: Para testar leitura de QR Codes, instale:")
    print("   pip install pyzbar opencv-python")
    print("   (Requer também libzbar no sistema: apt install libzbar0 ou brew install zbar)")

## 🎯 Próximos Passos

1. **Inserir QR Codes no LaTeX**: Os arquivos em `qr/generated/` estão prontos para uso
2. **Testar com celular**: Aponte a câmera para um QR Code e verifique o redirecionamento
3. **Personalizar URLs**: Edite `CHAPTERS` em `generate_qrcodes.py` para apontar para suas URLs
4. **Habilitar chat com IA**: Configure `QWEN_API_KEY` para ativar respostas dinâmicas

## ⚠️ Notas Importantes

- QR Codes contêm JSON compactado: escaneie com app que suporte leitura de payloads
- Para LaTeX, prefira formato PNG ou SVG
- Cache de respostas da IA é salvo em `~/.fen-trans/qwen_cache/`
- Modo offline sempre disponível como fallback

---
*Gerado em: ` + datetime.now().strftime("%Y-%m-%d %H:%M") + `*